In [ ]:
import matplotlib
matplotlib.use("Agg")  # Set matplotlib backend for non-interactive mode
import yaml
import pandas as pd
from pathlib import Path
from IPython.display import Image, display
from risk_validation.core.metrics.impl import pd as pd_metrics  # noqa: F401 - registers PD metrics
from risk_validation.core.services.pd.metrics_service import PDMetricsService
from risk_validation.core.utils.plots import plot_roc, plot_gini, plot_ks_cdf_with_maxgap
from risk_validation.core.utils.rag import rag_for_metric

# Load dataset
DATA_DIR = Path('..') / 'data'
df = pd.read_csv(DATA_DIR / 'sample.csv')

# Extract required columns and segregate DEV/VAL data
params = {
    'y_col': 'default_flag',
    'p_col': 'pred_br',
    'score_col': 'score_pd'
}
dev_year, val_year = 2019, 2020

# Filter data for DEV and VAL
available_years = df['score_year'].unique()
df_dev = df[df['score_year'] == dev_year]
df_val = df[df['score_year'] == val_year]

# Initialize the metrics service
service = PDMetricsService(['gini', 'auc_roc', 'ks'])

# Compute metrics for DEV and VAL
result_dev = service.compute(df_dev, params)
result_val = service.compute(df_val, params)

# Load configuration for RAG ratings
rag_path = Path('../configs/validation_rag.yaml')
with open(rag_path, 'r') as file:
    rag_config = yaml.safe_load(file)

# Compute and print RAG status for each metric
for metric_id in ['gini', 'auc_roc', 'ks']:
    dev_metric = result_dev[result_dev['metric_id'] == metric_id]
    val_metric = result_val[result_val['metric_id'] == metric_id]

    if not dev_metric.empty and not val_metric.empty:
        dev_value = dev_metric['value'].iloc[0]
        val_value = val_metric['value'].iloc[0]
        rag_result = rag_for_metric(metric_id, rag_config['pd'], dev_value, val_value)
        print(f"{metric_id.upper()} DEV={dev_value:.4f}, VAL={val_value:.4f}, RAG={rag_result['status']}")

# Generate and display plots
plot_dir = Path('plots')
plot_dir.mkdir(exist_ok=True)

# ROC Curve
roc_path = plot_roc(
    y_true=df_val['default_flag'],
    y_score=df_val['pred_br'],
    title='ROC Curve - Validation Data',
    out_path=plot_dir / 'roc_val.png'
)
display(Image(filename=str(roc_path)))

# Gini Plot
# Use plot_gini for a gains-like representation if needed

gini_path = plot_gini(
    y_true=df_val['default_flag'],
    y_score=df_val['pred_br'],
    title='Gini Plot - Validation Data',
    out_path=plot_dir / 'gini_val.png'
)
display(Image(filename=str(gini_path)))

# KS Statistic Plot
ks_path, _ = plot_ks_cdf_with_maxgap(
    y_true=df_val['default_flag'],
    y_score=df_val['pred_br'],
    title='KS CDF with Max Gap - Validation Data',
    out_path=plot_dir / 'ks_val.png'
)
display(Image(filename=str(ks_path)))